# HEAL-Summ-Lite: Lightweight Ethical Health Summarization Pipeline

## Architecture (5 Stages, Two-Shot Prompting)
```
Original Text (600-700 words, FULL — no pre-extraction)
  → [Stage 1] Two-Shot Phi-3 Mini Summarization  (1 LLM call, ~2-5 min)
  → [Stage 2] POST-PROCESSING (trim to 180 words, add caveat, source, break sentences)
  → [Stage 3] Readability Scoring (FKGL + FRE)    (Python, instant)
  → [Stage 4] 10 Rule-Based Heuristic Checks      (Python, instant)
  → [Stage 5] Tiered Human Review Decision         (Python, instant)
  → Results Tables + Severity Report
```

## Key Design Decisions
- **Two-shot prompting** — two SIMPLE examples teach the model short sentences and easy words
- **Post-processing** — enforce 180 word limit, add safety caveat, source attribution, break long sentences
- **Full text input** — no pre-extraction (TextRank/TF-IDF removed entities and lowered readability)
- **Sentence-count prompt** — "write 10-13 sentences" with 8-12 words each
- **10 deterministic heuristics** — instant, reproducible, auditable quality checks

In [1]:
!pip install requests textstat tabulate --quiet
print("✓ Dependencies installed.")

✓ Dependencies installed.


## Step 2: Imports & Configuration

In [2]:
import json
import re
import time
import difflib
import requests
import textstat
from tabulate import tabulate
from health_texts import HEALTH_TEXTS

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "phi3:mini"
SUMMARIZATION_TEMPERATURE = 0.3
MAX_FKGL_GRADE = 12
MIN_WORD_COUNT = 120
MAX_WORD_COUNT = 180
FRE_MINIMUM = 30.0

print(f"  Model: {MODEL_NAME}")
print(f"  Target: FKGL < {MAX_FKGL_GRADE}, FRE > {FRE_MINIMUM}")
print(f"  Word limit: {MIN_WORD_COUNT}-{MAX_WORD_COUNT} words")
print(f"  Total health-related texts available: {len(HEALTH_TEXTS)}")

  Model: phi3:mini
  Target: FKGL < 12, FRE > 30.0
  Word limit: 120-180 words
  Total health-related texts available: 5


## Step 3: SIMPLIFIED Two-Shot Examples & Prompt

**Why simplified examples?** The model mimics the STYLE of examples it sees:
- Old examples had 18-22 word sentences → Model generated complex output (FKGL 16+)
- New examples have 8-12 word sentences → Model should generate simpler output (FKGL < 10)

**Target readability:**
- Example 1: FKGL ~5, FRE ~77
- Example 2: FKGL ~3, FRE ~88

In [5]:
# =============================================================================
# SIMPLIFIED TWO-SHOT EXAMPLES (FKGL < 10, FRE > 50)
# =============================================================================

EXAMPLE_1_INPUT = """Asthma is a chronic respiratory condition that affects the airways in the lungs. During an asthma attack, the airways become inflamed and narrowed, making it difficult to breathe. Common triggers include allergens such as pollen and dust mites, cold air, physical exercise, and respiratory infections. According to WHO, approximately 262 million people worldwide were affected by asthma in 2019, and it caused 455,000 deaths that year. Asthma is the most common chronic disease among children. While asthma cannot be cured, proper management with inhaled corticosteroids and bronchodilators can control symptoms effectively. Regular monitoring of symptoms and lung function helps prevent severe attacks. Patients should avoid known triggers, follow their asthma action plan, get a flu vaccine each year, and consult a healthcare professional for personalized treatment."""

EXAMPLE_1_OUTPUT = """Asthma is a lung disease that makes breathing hard. During an attack, the airways swell and get narrow. Pollen, dust, cold air, and exercise can cause attacks. Lung infections can also trigger asthma symptoms. The WHO says 262 million people had asthma in 2019. That year, 455,000 people died from it worldwide. It is the most common long-term illness in kids. There is no cure for asthma yet. But medicines like inhalers can control the symptoms well. People with asthma should avoid things that trigger attacks. Checking symptoms often helps prevent serious problems. Getting a flu shot each year also helps. Talk to a doctor before making health choices."""

EXAMPLE_2_INPUT = """Tuberculosis (TB) is caused by bacteria called Mycobacterium tuberculosis that most often affect the lungs. TB is spread through the air when infected people cough, sneeze, or spit. About a quarter of the global population is estimated to have been infected with TB bacteria. The WHO reported 10.6 million new TB cases and 1.3 million deaths from TB in 2022. TB is preventable and curable. About 85% of people who develop TB disease can be successfully treated with a 6-month drug regimen. TB treatment has averted over 75 million deaths since the year 2000. Multidrug-resistant TB remains a public health crisis, with only about 2 in 5 affected people accessing treatment. The BCG vaccine is given to children in many countries to prevent severe forms of childhood TB. Early detection through diagnostic testing is essential because delayed treatment increases transmission risk. Consult a healthcare professional if you experience persistent cough, fever, or unexplained weight loss."""

EXAMPLE_2_OUTPUT = """TB is a lung infection caused by bacteria. It spreads through the air when sick people cough or sneeze. About 1 in 4 people in the world carry TB germs. In 2022, there were 10.6 million new TB cases globally. That same year, 1.3 million people died from TB. The good news is TB can be stopped and cured. About 85% of TB patients get well with 6 months of drugs. Since 2000, TB treatment has saved over 75 million lives. But drug-resistant TB is still a big problem. Only 2 in 5 people with this form get treatment. The BCG shot helps protect children from severe TB. Finding TB early through tests is very important. Talk to a doctor if you have a lasting cough or fever."""

# =============================================================================
# SIMPLIFIED SYSTEM PROMPT
# =============================================================================

SUMMARIZATION_SYSTEM_PROMPT = """You write health facts for people with basic reading skills.

STRICT RULES - YOU MUST FOLLOW ALL OF THESE:

SENTENCE LENGTH:
- Write 10 to 12 sentences total.
- Each sentence MUST have 8 to 12 words only.
- NEVER write a sentence longer than 15 words.
- If a sentence is too long, split it into two.

WORD LIMIT:
- Your summary MUST be between 120 and 170 words.
- Do NOT exceed 170 words.

WORD CHOICE:
- Use simple words that a child can read.
- Avoid big medical words when possible.
- BAD words: transmission, intervention, disproportionately, administered
- GOOD words: spread, help, mostly, given

MEDICAL TERMS:
- If you must use a medical term, explain it right after.
- Example: "ITNs are bed nets treated with bug spray."

LISTS:
- Never list more than 3 things in one sentence.

NUMBERS:
- Keep ALL numbers exactly as they appear.
- Do not round or change any numbers.

CONTENT:
- Cover the beginning, middle, AND end of the text.
- Do not add facts that are not in the text.

ENDING:
- End with: "Talk to a doctor before making health choices."

FORMAT:
- Write in plain paragraphs. No bullet points."""


def build_summarization_prompt(original_text):
    """Builds a two-shot prompt with simplified examples."""
    return f"""{SUMMARIZATION_SYSTEM_PROMPT}

Here are two examples of good health summaries:

EXAMPLE 1 INPUT:
{EXAMPLE_1_INPUT}

EXAMPLE 1 SUMMARY:
{EXAMPLE_1_OUTPUT}

EXAMPLE 2 INPUT:
{EXAMPLE_2_INPUT}

EXAMPLE 2 SUMMARY:
{EXAMPLE_2_OUTPUT}

Now summarize the following health text the same way — short sentences, simple words, 120-170 words:

INPUT TEXT:
{original_text}

SUMMARY:"""


def clean_incomplete_output(text):
    """Removes trailing incomplete sentence fragment."""
    text = text.strip()
    if text and text[-1] in '.!?':
        return text
    last_end = max(text.rfind('.'), text.rfind('!'), text.rfind('?'))
    if last_end > 0:
        return text[:last_end + 1]
    return text


def call_ollama(prompt, temperature):
    """Sends a prompt to Phi-3 Mini via Ollama local API."""
    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "top_p": 0.9,
            "num_predict": 250,  # Reduced to help with word limit
            "num_ctx": 4096,
            "num_thread": 4,
            "repeat_penalty": 1.1,
        },
    }
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=600)
        response.raise_for_status()
    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Cannot connect to Ollama. Make sure it is running.")
        return None
    except requests.exceptions.ReadTimeout:
        print("\n[ERROR] Timeout (>10 min). Close other apps to free RAM.")
        return None
    except requests.exceptions.HTTPError as e:
        print(f"\n[ERROR] Ollama error: {e}")
        return None
    try:
        result = response.json()
        return result.get("response", "").strip() or None
    except Exception as e:
        print(f"\n[ERROR] Invalid Ollama response: {e}")
        return None


# Quick connectivity test
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    models = [m['name'] for m in r.json().get('models', [])]
    print(f"✓ Ollama is running. Models: {models}")
except:
    print("⚠ Cannot connect to Ollama. Run: ollama serve")

✓ Ollama is running. Models: ['phi3:mini']


## Step 4: POST-PROCESSING (with Word Limit Enforcement)

Automatic fixes applied AFTER LLM generation:
1. **Enforce word limit** — Trim to 180 words max (keeping complete sentences)
2. **Add caveat** if treatments mentioned but no safety disclaimer
3. **Add source attribution** if missing (e.g., "According to WHO...")
4. **Break long sentences** to improve readability

In [6]:
# =============================================================================
# POST-PROCESSING: AUTOMATIC FIXES WITH WORD LIMIT ENFORCEMENT
# =============================================================================

def enforce_word_limit(text, max_words=180, min_words=120):
    """
    Trim text to max_words while keeping complete sentences.
    Prioritizes keeping the caveat sentence at the end.
    
    Args:
        text: The summary text
        max_words: Maximum allowed words (default 180)
        min_words: Minimum words to keep (default 120)
    
    Returns:
        Trimmed text with complete sentences
    """
    words = text.split()
    
    # If already within limit, return as-is
    if len(words) <= max_words:
        return text
    
    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    
    # Check if last sentence is a caveat (we want to keep it)
    caveat_keywords = ['doctor', 'consult', 'healthcare', 'medical', 'professional']
    last_sentence = sentences[-1] if sentences else ""
    has_caveat = any(kw in last_sentence.lower() for kw in caveat_keywords)
    
    if has_caveat and len(sentences) > 1:
        # Keep the caveat, trim from the middle/end of content
        caveat = sentences[-1]
        content_sentences = sentences[:-1]
        caveat_words = len(caveat.split())
        available_words = max_words - caveat_words
        
        # Build up content within word limit
        result_sentences = []
        current_words = 0
        
        for sent in content_sentences:
            sent_words = len(sent.split())
            if current_words + sent_words <= available_words:
                result_sentences.append(sent)
                current_words += sent_words
            else:
                # Check if we have enough words
                if current_words >= (min_words - caveat_words):
                    break
                # Need more content, include partial if necessary
                result_sentences.append(sent)
                current_words += sent_words
                break
        
        # Add caveat back
        result_sentences.append(caveat)
        return ' '.join(result_sentences)
    
    else:
        # No caveat at end, just trim from the end
        result_sentences = []
        current_words = 0
        
        for sent in sentences:
            sent_words = len(sent.split())
            if current_words + sent_words <= max_words:
                result_sentences.append(sent)
                current_words += sent_words
            else:
                if current_words >= min_words:
                    break
                result_sentences.append(sent)
                break
        
        return ' '.join(result_sentences)


def add_caveat_if_missing(summary, original):
    """
    Add safety caveat if treatments are mentioned but no disclaimer exists.
    """
    treatment_words = [
        'treatment', 'treated', 'medicine', 'drug', 'vaccine', 'therapy',
        'cure', 'medication', 'prescription', 'dose', 'pill', 'injection',
        'surgery', 'diagnosed', 'diagnosis', 'antiviral', 'antibiotic'
    ]
    caveat_phrases = [
        'talk to a doctor', 'consult a doctor', 'see a doctor',
        'healthcare professional', 'medical advice', 'speak to',
        'ask your doctor', 'consult', 'physician'
    ]
    
    has_treatment = any(word in original.lower() for word in treatment_words)
    has_caveat = any(phrase in summary.lower() for phrase in caveat_phrases)
    
    if has_treatment and not has_caveat:
        summary = summary.rstrip()
        if not summary.endswith('.'):
            summary += '.'
        summary += " Talk to a doctor before making health choices."
    
    return summary


def add_source_if_missing(summary, source_name):
    """
    Add source attribution at the beginning if missing.
    """
    source_map = {
        'World Health Organization': 'WHO',
        'Centers for Disease Control': 'CDC',
        'National Health Service': 'NHS',
        'Health Canada': 'Health Canada',
    }
    
    source_terms = ['WHO', 'CDC', 'NHS', 'Health Canada', 'World Health', 'Centers for Disease']
    has_source = any(term.lower() in summary.lower() for term in source_terms)
    
    if not has_source:
        short_source = None
        for full_name, short in source_map.items():
            if full_name in source_name or short in source_name:
                short_source = short
                break
        
        if short_source:
            first_char = summary[0].lower()
            rest = summary[1:]
            summary = f"According to the {short_source}, {first_char}{rest}"
    
    return summary


def break_long_sentences(text, max_words=15):
    """
    Break sentences longer than max_words at natural break points.
    """
    sentences = re.split(r'(?<=[.!?])\s+', text)
    result = []
    
    for sent in sentences:
        words = sent.split()
        
        if len(words) > max_words:
            break_patterns = [
                (', and ', '. '),
                (', but ', '. But '),
                (', which ', '. This '),
                ('; ', '. '),
            ]
            
            broken = False
            for pattern, replacement in break_patterns:
                if pattern in sent:
                    idx = sent.find(pattern)
                    part1 = sent[:idx].strip()
                    part2 = sent[idx + len(pattern):].strip()
                    
                    if len(part1.split()) >= 4 and len(part2.split()) >= 4:
                        if not part1.endswith('.'):
                            part1 += '.'
                        if part2:
                            part2 = part2[0].upper() + part2[1:]
                        result.append(part1)
                        result.append(part2)
                        broken = True
                        break
            
            if not broken:
                result.append(sent)
        else:
            result.append(sent)
    
    return ' '.join(result)


def post_process_summary(summary, original, source_name):
    """
    Apply all post-processing fixes to the summary.
    
    Order matters:
    1. Add caveat first (before trimming, so it's kept)
    2. Break long sentences
    3. Enforce word limit (keeps caveat at end)
    4. Add source attribution last (adds words at beginning)
    """
    # Step 1: Add caveat if missing
    summary = add_caveat_if_missing(summary, original)
    
    # Step 2: Break long sentences
    summary = break_long_sentences(summary, max_words=18)
    
    # Step 3: Enforce word limit (will keep caveat at end)
    summary = enforce_word_limit(summary, max_words=175, min_words=120)
    
    # Step 4: Add source (only if still under limit after adding)
    current_words = len(summary.split())
    if current_words <= 170:  # Leave room for source attribution (~5-6 words)
        summary = add_source_if_missing(summary, source_name)
    
    # Final check - hard limit at 180
    summary = enforce_word_limit(summary, max_words=180, min_words=120)
    
    return summary



✓ Post-processing functions loaded:
  - enforce_word_limit() — trims to 180 words max
  - add_caveat_if_missing()
  - add_source_if_missing()
  - break_long_sentences()

  Test: 108 words → 52 words (limit: 50)


## Step 5: Readability Scoring

In [7]:
def compute_readability(text):
    if not text or not text.strip():
        return {"fkgl": None, "fre": None}
    return {
        "fkgl": round(textstat.flesch_kincaid_grade(text), 1),
        "fre": round(textstat.flesch_reading_ease(text), 1),
    }

print("✓ Readability scoring ready.")

✓ Readability scoring ready.


## Step 6: 10 Rule-Based Heuristic Checks

| # | Check | What It Catches | Severity |
|---|-------|-----------------|----------|
| 1 | Numeric Coverage | Missing statistics | WARNING |
| 2 | Entity Coverage | Missing drug/org names | WARNING |
| 3 | Compression Ratio | Summary too long/short | WARNING |
| 4 | Caveat Presence | Treatment without disclaimer | CRITICAL |
| 5 | Negation Flip | "does NOT" → "does" | CRITICAL |
| 6 | Hedging Preservation | "may" → definitive | WARNING |
| 7 | Source Attribution | Missing WHO/CDC | INFO |
| 8 | Sentence Count | Too few/many sentences | WARNING |
| 9 | Semantic Similarity | Hallucination check | WARNING |
| 10 | Sensitive Topics | Missing crisis info | CRITICAL |

In [8]:
# --- HELPER: Extract numbers from text ---
def extract_numbers(text):
    raw = re.findall(r'\d[\d,]*\.?\d*%?', text)
    return set(num.replace(",", "") for num in raw)

# --- HELPER: Extract key entities ---
def extract_key_entities(text):
    entities = set()
    orgs = ["WHO", "CDC", "NHS", "World Health Organization",
            "Centers for Disease Control", "National Health Service"]
    for org in orgs:
        if org.lower() in text.lower():
            entities.add(org)
    drugs = re.findall(
        r'\b(vaccine|ITNs?|IRS|insulin|ACT|BCG|antibiotic|antiviral)\b',
        text, re.IGNORECASE)
    for drug in drugs:
        entities.add(drug.lower())
    diseases = re.findall(
        r'\b(diabetes|malaria|influenza|flu|TB|tuberculosis|asthma|HIV|AIDS)\b',
        text, re.IGNORECASE)
    for disease in diseases:
        entities.add(disease.lower())
    return entities

print("✓ Helper functions loaded.")

✓ Helper functions loaded.


In [9]:
# CHECK 1: Numeric Coverage
def check_numeric_coverage(original, summary):
    orig_nums = extract_numbers(original)
    summ_nums = extract_numbers(summary)
    if not orig_nums:
        return {"status": "PASS", "severity": "INFO", "reason": "No numbers in original", "detail": {}}
    missing = orig_nums - summ_nums
    coverage = round((len(orig_nums) - len(missing)) / len(orig_nums) * 100, 1)
    if coverage < 30:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {coverage}% of numbers preserved", "detail": {"missing": list(missing)[:5]}}
    return {"status": "PASS", "severity": "INFO", "reason": f"{coverage}% of numbers preserved", "detail": {}}

# CHECK 2: Entity Coverage
def check_entity_coverage(original, summary):
    orig_ents = extract_key_entities(original)
    summ_ents = extract_key_entities(summary)
    if not orig_ents:
        return {"status": "PASS", "severity": "INFO", "reason": "No key entities detected", "detail": {}}
    missing = orig_ents - summ_ents
    coverage = round((len(orig_ents) - len(missing)) / len(orig_ents) * 100, 1)
    if coverage < 40:
        return {"status": "FLAG", "severity": "WARNING",
                "reason": f"Only {coverage}% of entities preserved", "detail": {"missing": list(missing)[:5]}}
    return {"status": "PASS", "severity": "INFO", "reason": f"{coverage}% of entities preserved", "detail": {}}

# CHECK 3: Compression Ratio
def check_compression_ratio(original, summary):
    orig_words = len(original.split())
    summ_words = len(summary.split())
    if orig_words == 0:
        return {"status": "PASS", "severity": "INFO", "reason": "Empty original", "detail": {}}
    ratio = round(summ_words / orig_words, 2)
    if ratio > 0.5:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"Ratio {ratio} — summary too long", "detail": {}}
    if ratio < 0.15:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"Ratio {ratio} — may have lost content", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": f"Ratio {ratio} — acceptable", "detail": {}}

# CHECK 4: Caveat Presence
def check_caveat_presence(original, summary):
    treatment_keywords = ["treatment", "medicine", "drug", "vaccine", "therapy", "cure"]
    caveat_phrases = ["doctor", "consult", "healthcare", "medical advice", "professional"]
    has_treatment = any(kw in original.lower() for kw in treatment_keywords)
    has_caveat = any(phrase in summary.lower() for phrase in caveat_phrases)
    if has_treatment and not has_caveat:
        return {"status": "FLAG", "severity": "CRITICAL", "reason": "Treatment but NO caveat", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": "Caveat present or not needed", "detail": {}}

# CHECK 5: Negation Flip
def check_negation_flip(original, summary):
    # Simplified check
    critical_phrases = ["does not spread", "not contagious", "cannot be cured", "no cure"]
    for phrase in critical_phrases:
        if phrase in original.lower():
            # Check if the non-negated version appears in summary without negation
            positive = phrase.replace("not ", "").replace("no ", "").replace("cannot ", "can ")
            if positive in summary.lower() and phrase not in summary.lower():
                return {"status": "FLAG", "severity": "CRITICAL", "reason": f"Possible flip: '{phrase}'", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": "No negation flips detected", "detail": {}}

# CHECK 6: Hedging Preservation
def check_hedging_preservation(original, summary):
    hedging = ["may", "might", "could", "possibly", "estimated", "approximately"]
    orig_hedges = [h for h in hedging if h in original.lower()]
    if not orig_hedges:
        return {"status": "PASS", "severity": "INFO", "reason": "No hedging in original", "detail": {}}
    missing = [h for h in orig_hedges if h not in summary.lower()]
    if len(missing) > len(orig_hedges) * 0.7:
        return {"status": "FLAG", "severity": "WARNING", "reason": "Most hedging removed", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": "Hedging preserved", "detail": {}}

# CHECK 7: Source Attribution
def check_source_attribution(original, summary, source_name):
    source_terms = ["who", "cdc", "nhs", "world health", "centers for disease"]
    has_source = any(term in summary.lower() for term in source_terms)
    if not has_source and any(term in source_name.lower() for term in source_terms):
        return {"status": "FLAG", "severity": "INFO", "reason": f"Source not mentioned", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": "Source attribution OK", "detail": {}}

# CHECK 8: Sentence Count
def check_sentence_count(summary):
    sentences = re.split(r'(?<=[.!?])\s+', summary.strip())
    count = len(sentences)
    if count < 4:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"Only {count} sentences", "detail": {}}
    if count > 15:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"{count} sentences — too many", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": f"{count} sentences — OK", "detail": {}}

# CHECK 9: Semantic Similarity
def check_semantic_similarity(original, summary):
    orig_clean = re.sub(r'\s+', ' ', original.lower().strip())
    summ_clean = re.sub(r'\s+', ' ', summary.lower().strip())
    similarity = round(difflib.SequenceMatcher(None, orig_clean, summ_clean).ratio() * 100, 1)
    if similarity < 8:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"Similarity {similarity}% — very different", "detail": {}}
    if similarity > 85:
        return {"status": "FLAG", "severity": "WARNING", "reason": f"Similarity {similarity}% — too similar", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": f"Similarity {similarity}%", "detail": {}}

# CHECK 10: Sensitive Topics
def check_sensitive_topics(original, summary):
    sensitive = ["suicide", "self-harm", "overdose", "crisis"]
    has_sensitive = any(kw in original.lower() for kw in sensitive)
    if not has_sensitive:
        return {"status": "PASS", "severity": "INFO", "reason": "No sensitive topics", "detail": {}}
    resources = ["988", "911", "helpline", "hotline", "emergency"]
    has_resources = any(r in summary.lower() for r in resources)
    if not has_resources:
        return {"status": "FLAG", "severity": "CRITICAL", "reason": "Sensitive topic, no crisis info", "detail": {}}
    return {"status": "PASS", "severity": "INFO", "reason": "Crisis resources present", "detail": {}}

# RUN ALL 10 HEURISTICS
def run_all_heuristics(original, summary, source_name):
    return {
        "numeric_coverage":     check_numeric_coverage(original, summary),
        "entity_coverage":      check_entity_coverage(original, summary),
        "compression_ratio":    check_compression_ratio(original, summary),
        "caveat_presence":      check_caveat_presence(original, summary),
        "negation_flip":        check_negation_flip(original, summary),
        "hedging_preservation": check_hedging_preservation(original, summary),
        "source_attribution":   check_source_attribution(original, summary, source_name),
        "sentence_count":       check_sentence_count(summary),
        "semantic_similarity":  check_semantic_similarity(original, summary),
        "sensitive_topics":     check_sensitive_topics(original, summary),
    }

print("✓ All 10 heuristic checks loaded.")

✓ All 10 heuristic checks loaded.


## Step 7: Tiered Human Review

| Tier | Condition | Action |
|------|-----------|--------|
| **CRITICAL** | Any CRITICAL flag | Always escalate |
| **ESCALATE** | 2+ WARNING flags | Escalate |
| **REVIEW** | 1 WARNING flag | Soft flag |
| **APPROVED** | No flags | Safe to publish |

In [10]:
def decide_human_review(word_count, readability, heuristics):
    critical_reasons = []
    warning_reasons = []
    info_notes = []

    for check_name, result in heuristics.items():
        if result["status"] == "FLAG":
            severity = result.get("severity", "WARNING")
            tag = f"[{severity}] {check_name}: {result['reason']}"
            if severity == "CRITICAL":
                critical_reasons.append(tag)
            elif severity == "WARNING":
                warning_reasons.append(tag)
            else:
                info_notes.append(tag)

    # Readability checks
    if readability.get("fkgl") and readability["fkgl"] > MAX_FKGL_GRADE:
        warning_reasons.append(f"[WARNING] FKGL {readability['fkgl']} > {MAX_FKGL_GRADE}")
    if readability.get("fre") and readability["fre"] < FRE_MINIMUM:
        warning_reasons.append(f"[WARNING] FRE {readability['fre']} < {FRE_MINIMUM}")

    # Word count check - now should always pass due to post-processing
    if word_count < MIN_WORD_COUNT:
        warning_reasons.append(f"[WARNING] {word_count} words < {MIN_WORD_COUNT}")
    elif word_count > MAX_WORD_COUNT:
        warning_reasons.append(f"[WARNING] {word_count} words > {MAX_WORD_COUNT}")

    all_reasons = critical_reasons + warning_reasons + info_notes

    if critical_reasons:
        return "CRITICAL", all_reasons
    elif len(warning_reasons) >= 2:
        return "ESCALATE", all_reasons
    elif len(warning_reasons) == 1:
        return "REVIEW", all_reasons
    else:
        return "APPROVED", info_notes

print(" Tiered human review ready.")

✓ Tiered human review ready.


## Step 8: Main Pipeline (5 Stages)

In [11]:
def process_single_text(health_text):
    """Full HEAL-Summ-Lite pipeline with word limit enforcement."""
    text_id = health_text["id"]
    source = health_text["source"]
    title = health_text["title"]
    original = health_text["text"]

    print(f"\n{'='*70}")
    print(f"  Processing: {text_id} — {title}")
    print(f"  Source: {source}")
    print(f"  Original: {len(original.split())} words")
    print(f"{'='*70}")

    # --- Stage 1: Summarization ---
    print("\n  [1/5] Two-shot summarization (Phi-3 Mini)...")
    start_time = time.time()
    summary_prompt = build_summarization_prompt(original)
    raw_summary = call_ollama(summary_prompt, temperature=SUMMARIZATION_TEMPERATURE)
    if not raw_summary:
        print(" Summary generation failed.")
        return None
    gen_time = round(time.time() - start_time, 1)
    raw_summary = clean_incomplete_output(raw_summary)
    raw_word_count = len(raw_summary.split())
    print(f"        ✓ Raw: {raw_word_count} words in {gen_time}s")

    # --- Stage 2: POST-PROCESSING ---
    print("  [2/5] Post-processing (word limit, caveat, source)...")
    summary = post_process_summary(raw_summary, original, source)
    word_count = len(summary.split())
    
    changes = []
    if raw_word_count > 180 and word_count <= 180:
        changes.append(f"trimmed {raw_word_count}→{word_count} words")
    if "Talk to a doctor" in summary and "Talk to a doctor" not in raw_summary:
        changes.append("added caveat")
    if "According to" in summary and "According to" not in raw_summary:
        changes.append("added source")
    
    if changes:
        print(f"        ✓ {', '.join(changes)}")
    print(f"        ✓ Final: {word_count} words (limit: {MIN_WORD_COUNT}-{MAX_WORD_COUNT})")

    # --- Stage 3: Readability ---
    print("  [3/5] Readability scoring...")
    readability = compute_readability(summary)
    fkgl_ok = "✓" if readability.get('fkgl', 99) <= MAX_FKGL_GRADE else "⚠"
    fre_ok = "✓" if readability.get('fre', 0) >= FRE_MINIMUM else "⚠"
    print(f"        {fkgl_ok} FKGL: {readability.get('fkgl')} (target: ≤{MAX_FKGL_GRADE})")
    print(f"        {fre_ok} FRE:  {readability.get('fre')} (target: ≥{FRE_MINIMUM})")

    # --- Stage 4: Heuristics ---
    print("  [4/5] Running 10 heuristic checks...")
    heuristics = run_all_heuristics(original, summary, source)
    flags = sum(1 for v in heuristics.values() if v["status"] == "FLAG")
    passes = sum(1 for v in heuristics.values() if v["status"] == "PASS")
    print(f"        Results: {passes} PASS, {flags} FLAG")

    # --- Stage 5: Human Review ---
    print("  [5/5] Human review decision...")
    decision, review_reasons = decide_human_review(word_count, readability, heuristics)
    print(f"        Decision: {decision}")

    return {
        "id": text_id, "source": source, "title": title,
        "original_word_count": len(original.split()),
        "raw_summary": raw_summary, "raw_word_count": raw_word_count,
        "summary": summary, "summary_word_count": word_count,
        "fkgl": readability.get("fkgl"), "fre": readability.get("fre"),
        "heuristics": heuristics, "decision": decision,
        "review_reasons": review_reasons, "generation_time": gen_time,
    }

print(" Pipeline ready (with word limit enforcement).")

✓ Pipeline ready (with word limit enforcement).


## Step 9: Display Results

In [22]:
def display_results_table(results):
    print("\n" + "=" * 90)
    print("  HEAL-Summ-Lite — RESULTS")
    print("=" * 90)
    # Table 1: Overview
    print("\n  [TABLE 1] Summary Overview")
    print("  " + "-" * 80)
    t1_data = []
    for r in results:
        t1_data.append([
            r["id"],
            r["title"][:22] + ".." if len(r["title"]) > 22 else r["title"],
            r["original_word_count"],
            r["summary_word_count"],
            r["fkgl"],
            r["fre"],
            r["decision"],
        ])
    h1 = ["ID", "Title", "Orig", "Words", "FKGL", "FRE", "Decision"]
    print(tabulate(t1_data, headers=h1, tablefmt="grid"))
    # Flagged summaries
    flagged = [r for r in results if r["decision"] in ("CRITICAL", "ESCALATE")]
    if flagged:
        print(f"\n  FLAGGED: {len(flagged)} of {len(results)}")
        for r in flagged:
            print(f"    [{r['decision']}] {r['id']}")
    else:
        print("All summaries within acceptable range!")
    # Print summaries
    print("\n" + "=" * 90)
    print("  GENERATED SUMMARIES")
    print("=" * 90)
    for r in results:
        print(f"\n  --- {r['id']}: {r['title']} ---")
        print(f"  [{r['summary_word_count']} words | FKGL: {r['fkgl']} | FRE: {r['fre']} | {r['decision']}]")
        print(f"\n  {r['summary']}")
print("Display functions ready.")


Display functions ready.


## 🚀 Step 10: Run the Pipeline

In [23]:
print("=" * 60)
print("  HEAL-Summ-Lite: Select a Health Text")
print("=" * 60)
print()

for i, ht in enumerate(HEALTH_TEXTS):
    wc = len(ht["text"].split())
    print(f"  [{i+1}] {ht['id']} — {ht['title'][:40]}")
    print(f"      Source: {ht['source']} | Words: {wc}")
    print()

print(f"  [A] Run ALL {len(HEALTH_TEXTS)} texts")
print()

choice = input("  Select (1-5, A=all): ").strip()

if choice.upper() == "A":
    selected = HEALTH_TEXTS
elif choice.isdigit() and 1 <= int(choice) <= len(HEALTH_TEXTS):
    selected = [HEALTH_TEXTS[int(choice) - 1]]
else:
    selected = [HEALTH_TEXTS[0]]

# Run pipeline
results = []
for health_text in selected:
    result = process_single_text(health_text)
    if result:
        results.append(result)

# Display
if results:
    display_results_table(results)
    with open("heal_summ_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    print("\n  ✓ Results saved to heal_summ_results.json")

  HEAL-Summ-Lite: Select a Health Text

  [1] TEXT_01 — Diabetes – Key Facts and Prevention
      Source: World Health Organization (WHO) | Words: 605

  [2] TEXT_02 — About Influenza (Flu) – Symptoms, Spread
      Source: Centers for Disease Control and Prevention (CDC) | Words: 607

  [3] TEXT_03 — High Blood Pressure (Hypertension) – Cau
      Source: National Health Service (NHS), UK | Words: 638

  [4] TEXT_04 — Malaria – Key Facts, Prevention, and Tre
      Source: World Health Organization (WHO) | Words: 657

  [5] TEXT_05 — Mental Health – Understanding and Coping
      Source: Centers for Disease Control and Prevention (CDC) | Words: 693

  [A] Run ALL 5 texts



  Select (1-5, A=all):  4



  Processing: TEXT_04 — Malaria – Key Facts, Prevention, and Treatment
  Source: World Health Organization (WHO)
  Original: 657 words

  [1/5] Two-shot summarization (Phi-3 Mini)...
        ✓ Raw: 156 words in 311.5s
  [2/5] Post-processing (word limit, caveat, source)...
        ✓ added caveat, added source
        ✓ Final: 168 words (limit: 120-180)
  [3/5] Readability scoring...
        ✓ FKGL: 12.0 (target: ≤12)
        ✓ FRE:  46.5 (target: ≥30.0)
  [4/5] Running 10 heuristic checks...
        Results: 7 PASS, 3 FLAG
  [5/5] Human review decision...
        Decision: ESCALATE

  HEAL-Summ-Lite — RESULTS

  [TABLE 1] Summary Overview
  --------------------------------------------------------------------------------
+---------+--------------------------+--------+---------+--------+-------+------------+
| ID      | Title                    |   Orig |   Words |   FKGL |   FRE | Decision   |
+=========+==========================+========+=========+========+=======+============+
| TEX

In [24]:
# Run ALL 5 texts through the pipeline
results = []
total_start = time.time()

for i, health_text in enumerate(HEALTH_TEXTS):
    print(f"\n{'#'*70}")
    print(f"  TEXT {i+1} of {len(HEALTH_TEXTS)}")
    print(f"{'#'*70}")
    result = process_single_text(health_text)
    if result:
        results.append(result)

total_time = round(time.time() - total_start, 1)
print(f"\n\n{'='*70}")
print(f"  ALL DONE — {len(results)} texts processed in {total_time}s")
print(f"{'='*70}")

# Display results
if results:
    display_results_table(results)
    with open("heal_summ_results.json", "w") as f:
        json.dump(results, f, indent=2, default=str)
    print("\n  Results saved to heal_summ_results.json")


######################################################################
  TEXT 1 of 5
######################################################################

  Processing: TEXT_01 — Diabetes – Key Facts and Prevention
  Source: World Health Organization (WHO)
  Original: 605 words

  [1/5] Two-shot summarization (Phi-3 Mini)...
        ✓ Raw: 147 words in 256.5s
  [2/5] Post-processing (word limit, caveat, source)...
        ✓ added caveat, added source
        ✓ Final: 158 words (limit: 120-180)
  [3/5] Readability scoring...
        ✓ FKGL: 9.4 (target: ≤12)
        ✓ FRE:  62.6 (target: ≥30.0)
  [4/5] Running 10 heuristic checks...
        Results: 7 PASS, 3 FLAG
  [5/5] Human review decision...
        Decision: ESCALATE

######################################################################
  TEXT 2 of 5
######################################################################

  Processing: TEXT_02 — About Influenza (Flu) – Symptoms, Spread, and Prevention
  Source: Centers for Dise